In [4]:
import os
from pathlib import Path
import sys


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가합니다.
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

os.environ.setdefault("LANGSMITH_TRACING", "false")

'true'

In [5]:
from langchain_community.document_loaders import PyPDFLoader

file_path = str(PROJECT_ROOT / "data/raw/pdf/0417_kca_split.pdf")
docs = PyPDFLoader(file_path=file_path).load()
print(f"페이지 수: {len(docs)}")

페이지 수: 57


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("kca_rag_index")
print(f"완료: {vectorstore.index.ntotal}개 벡터 저장")

완료: 57개 벡터 저장


In [7]:
vectorstore = FAISS.load_local("kca_rag_index", embeddings, allow_dangerous_deserialization=True)

In [8]:
retriever = vectorstore.as_retriever()
query = "디지털 결제 수단 이용률"   # ← 원하는 질문으로 변경

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):
    print(f"[{i}위] p.{doc.metadata['page']+1}")
    print(doc.page_content[:300])
    print("=" * 60)

[1위] p.3
PART 1_제5장 2023 가계소비 현황과 인식 763
【그림 5-2-1】 디지털결제 수단 이용률
(Base: 전체, 단위: %)
【그림 5-2-2】 디지털결제 수단 으로 주로 소비한 분야(1 ＋2＋3순위 기준)
(Base: 디지털결제 수단 이용자, n=4,993명, 단위: %)
[2위] p.5
PART 1_제5장 2023 가계소비 현황과 인식 765
【그림 5-2-4】 연령별 디지털결제 수단 으로 주로 소비한 품목(1 ＋2＋3순위 기준)
(Base: 디지털결제 수단 이용자, n=4,993명, 단위: %)
[3위] p.1
PART 1_제5장 2023 가계소비 현황과 인식 761
제2절 비대면 디지털시대 가계소비 모습
지표 
디지털 소비생활20) 디지털 소비와 결제수단 이용 
현황10-64
◦ 디지털결제 수 단 이용률은 49.9%로, 2명 중 1명 수준이며, 전자상거래 경험자의 디지털 결
제수 단 이용률은 65.3%이며, 코로나19 이 후 소비혼란 경험자의 이용률은 50.9%로 나타남
◦ 주 소비 분야는 식품·외식 92.7%, 의류 36.4%, 문화·여가 18.8%, 생활위생· 미용 18.0% 
등의 순으로 나타남
◦ 온라인으로 소비한 분야 조사한 결
[4위] p.4
764 2023 한국의 소비생활지표
【그림 5-2-3】 전자상거래 경험별 디지털결제 수단 으로 주로 소비한 품목
(1＋2＋3순위 기준)
(Base: 디지털결제 수단 이용자, n=4,993명, 단위: %)
